# tp-12b 改善したオッズ予測（tp-20 の V8）での馬券最適化（検証年のみ）

tp-20 で、オッズ予測は市場確率の誤差が基準（固定パラメータの LightGBM 1 本）から約 14% 減った（V8 = 探索パラメータの LightGBM と CatBoost の平均、特徴量に追い切り評価・厩舎コメント・血統×条件を追加）。ここでは、**単勝の予測オッズを V8 に替えて**、tp-12 と同じ馬券最適化（RL 方策、基準、2 段階確認）をやり直し、回収率が変わるかを見る。複勝オッズの予測は tp-12 と同じ。

- 評価は fold A = 学習 2022 → 評価 2023、fold B = 学習 2022〜2023 → 評価 2024 の**検証年のみ**。**test 2025 は使わない**（tp-12 で評価済み。やり直さない）。
- オッズ予測の特徴量にオッズ・人気は含めない（列名検査）。V8 の 2023 年分は tp-20 でパラメータ選択に使った年（楽観側）。
- tp-12（`pred_v1`）の fold A・B の結果（`pred_odds/pred_v1/valid_compare.csv`）と並べて比べる。


In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time, gc
from datetime import datetime
import numpy as np
import pandas as pd
import mlflow
import lightgbm as lgb

from src.pipeline.betting import policy as pol
from src.pipeline.betting import odds_model_eval as oe
from src.pipeline.betting import pred_odds as po
from src.pipeline.betting import ticket_env as te
from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
TAG = os.environ.get("TP12B_TAG", "pred_v8")
SMOKE = os.environ.get("TP12B_SMOKE") == "1"
OUT = BET / "pred_odds" / TAG
OUT.mkdir(parents=True, exist_ok=True)
LOCK = BET / "pred_odds" / f"TEST2025_EVALUATED_{TAG}.json"
THREADS = int(os.environ.get("TP12B_THREADS", max(1, min(14, detect_compute(reserve_cores=6).threads))))
SEEDS = (0,) if SMOKE else (0, 1, 2)
FINAL_SEEDS = (0, 1) if SMOKE else (0, 1, 2, 3, 4)
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-betting-predicted-odds")
require_tracking_server()
YEARS = ["2022", "2023", "2024"]
wp = pd.read_parquet(BET / "odds_win_place.parquet"); wp = wp[wp["race_id"].str[:4].isin(YEARS) & (wp["win_odds"] > 0)].reset_index(drop=True)
pay = pd.read_parquet(BET / "payoff.parquet"); pay = pay[pay["race_id"].str[:4].isin(YEARS)].reset_index(drop=True)
cols = ["race_id", "horse_id", "year", "p_win", "p_top2", "p_top3"]
pred_all = pd.concat([pd.read_parquet(WORK / "oof/model_oof/ensemble_v2.parquet")[cols], pd.read_parquet(WORK / "final_eval_v2/test2025_predictions.parquet")[cols]], ignore_index=True)
pred_all = pred_all[pred_all["year"].between(2022, 2024)]
BEST8 = json.loads((BET / "rl/ens_v2/best_config.json").read_text(encoding="utf-8"))["params"]
CMP = BET / "rl_compare/cmp_v2"
TDIR = BET / "tickets/cmp_v2_K12_v2"
TICK_ACT = {y: pd.read_parquet(TDIR / f"{y}.parquet") for y in YEARS}          # 確定オッズの買い目（参考と、2段階の確認に使う）
AL = pd.read_csv(CMP / "alpha_mle.csv", index_col=0)
AL2, AL3 = float(AL.loc["最尤推定（2022 年）", "α2"]), float(AL.loc["最尤推定（2022 年）", "α3"])
CONSTS = po.pair_constants(TICK_ACT["2022"])
print("α2, α3 =", round(AL2, 3), round(AL3, 3), "| c_type =", {k: round(v, 3) for k, v in CONSTS.items()})

## 1. オッズ予測（ローリング、オッズ関連情報なし）

In [ ]:
wp_all = pd.read_parquet(BET / "odds_win_place.parquet"); wp_all = wp_all[wp_all["win_odds"] > 0]
D = tp05_data.load_dataset(WORK)
df = D["df"]
feats = D["FEATS"]["base+S+C1"]
EP, _EX, _PED = tp05_data.load_extras_ped(df, WORK)
po.assert_no_odds_features(list(feats) + list(EP.columns))            # オッズ・人気に関わる列が混ざっていれば停止
T = df[["race_id", "horse_id", "date"]].reset_index().merge(wp_all[["race_id", "horse_id", "win_odds", "place_odds_min"]], on=["race_id", "horse_id"], how="inner")
X = df.loc[T["index"], feats].reset_index(drop=True)
X_ep = pd.concat([X, EP.loc[T["index"]].reset_index(drop=True)], axis=1)
dates = pd.to_datetime(T["date"]); yr = dates.dt.year.to_numpy()
race_all = T["race_id"].to_numpy()
p_mkt = oe.market_prob(race_all, T["win_odds"].to_numpy())
ylog = {"win": np.log(T["win_odds"].to_numpy()), "place": np.log(np.clip(T["place_odds_min"].fillna(1.1).to_numpy(), 1.0, None)), "pm": np.log(p_mkt)}
P = {"objective": "regression", "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 50, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
     "verbosity": -1, "num_threads": THREADS, "seed": 0, "deterministic": True, "force_row_wise": True}
P8 = dict(P, learning_rate=0.02, feature_fraction=0.7)            # tp-20 で選んだ探索パラメータ（num_leaves=63, min_data_in_leaf=50, lambda_l2=1.0）
HL = None
rep = WORK / "odds_drift/report.json"
if rep.exists():
    prop = json.loads(rep.read_text(encoding="utf-8")).get("proposal")
    print("tp-10 の提案:", prop)
    HL = {"半減期1年": 1.0, "半減期2年": 2.0}.get(prop)
try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None
cat_cols = [c for c in X_ep.columns if str(X_ep[c].dtype) in ("category", "object")]
Xc = X_ep.copy()
for c in cat_cols:
    Xc[c] = Xc[c].astype(str)
rows, preds = [], []
for Y in ([2023, 2024] if SMOKE else [2022, 2023, 2024]):
    t0 = time.time()
    tr = np.flatnonzero(dates < pd.Timestamp(f"{Y}-01-01") - pd.Timedelta(days=10)); te_ = np.flatnonzero(yr == Y)
    out = T.iloc[te_][["race_id", "horse_id"]].copy()
    # 単勝 = V8: 探索パラメータの LightGBM（前年で早期終了）と CatBoost の log 市場確率の平均
    tr_a, va = tr[yr[tr] < Y - 1], tr[yr[tr] == Y - 1]
    b0 = lgb.train(P8, lgb.Dataset(X_ep.iloc[tr_a], label=ylog["pm"][tr_a]), num_boost_round=3000, valid_sets=[lgb.Dataset(X_ep.iloc[va], label=ylog["pm"][va])], callbacks=[lgb.early_stopping(80, verbose=False)])
    b = lgb.train(P8, lgb.Dataset(X_ep.iloc[tr], label=ylog["pm"][tr]), num_boost_round=int(max(50, b0.best_iteration * 1.1)))
    z = b.predict(X_ep.iloc[te_])
    if CatBoostRegressor is not None:
        m = CatBoostRegressor(iterations=300 if SMOKE else 1500, learning_rate=0.06, depth=8, loss_function="RMSE", thread_count=THREADS, random_seed=0, verbose=0, cat_features=cat_cols)
        m.fit(Xc.iloc[tr], ylog["pm"][tr]); z = (z + m.predict(Xc.iloc[te_])) / 2
    out["win_raw"] = 1.0 / np.exp(z)                                  # 暗黙の確率の逆数（オッズ相当）。レース内の正規化は後で行う
    # 複勝 = tp-12 と同じ（固定パラメータの LightGBM、base+S+C1）
    w = (0.5 ** (((Y - 1) - yr[tr]) / HL)) if HL else None
    bp = lgb.train(P, lgb.Dataset(X.iloc[tr], label=ylog["place"][tr], weight=w), num_boost_round=500)
    out["place_raw"] = np.exp(bp.predict(X.iloc[te_]))
    for k in ("win", "place"):
        r = ylog[k][te_] - np.log(out[f"{k}_raw"].to_numpy())
        rows.append({"検証年": Y, "対象": k, "rmse_log": float(np.sqrt((r ** 2).mean())), "偏り": float(r.mean()), "sec": time.time() - t0})
    preds.append(out); gc.collect()
    print(Y, f"{time.time() - t0:.0f}s", flush=True)
PO = pd.concat(preds, ignore_index=True)
PO["win_pred"] = po.normalize_win_odds(PO["race_id"].to_numpy(), PO["win_raw"].to_numpy())
PO["place_pred"] = np.clip(PO["place_raw"], 1.0, None)
acc = pd.DataFrame(rows); display(acc.round(4)); acc.to_csv(OUT / "odds_prediction_accuracy.csv", index=False)
chk = PO.merge(wp[["race_id", "horse_id", "win_odds", "place_odds_min"]], on=["race_id", "horse_id"])
print("正規化後の単勝: 対数 RMSE", round(float(np.sqrt(((np.log(chk.win_odds) - np.log(chk.win_pred)) ** 2).mean())), 4), "/ 中央の予測÷実際", round(float((chk.win_pred / chk.win_odds).median()), 3))


## 2. 予測オッズで判断する買い目の表
実際のオッズは使わない（馬の出走確認と払戻の精算にだけ使う）。

In [ ]:
TP = {}
TD = BET / "tickets" / f"{TAG}_K12"
TD.mkdir(parents=True, exist_ok=True)
for y in (["2023", "2024"] if SMOKE else YEARS):
    f = TD / f"{y}.parquet"; mf = TD / f"{y}.json"
    want = {"al": [round(AL2, 4), round(AL3, 4)], "consts": {k: round(v, 4) for k, v in CONSTS.items()}, "hl": HL, "ver": 1}
    if f.exists() and mf.exists() and json.loads(mf.read_text()) == want:
        TP[y] = pd.read_parquet(f); print(y, "キャッシュ", len(TP[y])); continue
    t0 = time.time()
    ids = set(pred_all.loc[pred_all["year"] == int(y), "race_id"])
    d = te.build_tickets_predicted_parallel(pred_all[pred_all["race_id"].isin(ids)], wp[wp["race_id"].isin(ids)], pay[pay["race_id"].isin(ids)], PO[PO["race_id"].isin(ids)], CONSTS, (AL2, AL3), K=12, procs=min(THREADS, 6), chunks=40)
    d.to_parquet(f, index=False); mf.write_text(json.dumps(want)); TP[y] = d
    print(f"{y}: {d['race_id'].nunique():,} / {len(ids):,} レース、{len(d):,} 点 ({time.time() - t0:.0f}s)", flush=True)

## 3. 戦略の比較
RL は tp-08 の最良設定（賭け金の下限 0.5）。確率はモデル単独、オッズは予測。

In [ ]:
def rl_alloc(tr_years, te_year, seeds):
    dtr = pd.concat([TP[y] for y in tr_years], ignore_index=True)
    dte = TP[te_year].reset_index(drop=True)
    pk_tr, mu, sd = pol.pack(dtr)
    pk_te, _, _ = pol.pack(dte, mu, sd)
    ws = []
    for s in seeds:
        c = pol.TrainConfig(**{**BEST8, "seed": s, "threads": min(THREADS, 4)})
        net, _ = pol.train_policy(pk_tr, c)
        ws.append(pol.predict_alloc(net, pk_te, c.max_ticket))
    return pk_te, dte, np.mean(ws, axis=0)


def baselines(pk, dte):
    ev = pol.pack_column(pk, dte.assign(_ev=dte["p_comb"] * dte["odds"]), "_ev")
    return {
        "モデル確率1位: 単勝": pol.alloc_top_p(pk, "tansho"), "モデル確率1位: 複勝": pol.alloc_top_p(pk, "fukusho"), "モデル確率1位: 馬連": pol.alloc_top_p(pk, "umaren"),
        "予測EV上位3点（参加100%）": pol.alloc_topk_score(pk, ev, 3, 1.0), "予測EV上位3点（参加50%）": pol.alloc_topk_score(pk, ev, 3, 0.5), "ランダム1点": pol.alloc_random(pk),
    }


def confirm(pk, dte, w, year, sigma, tau, seed=0):
    """2 段階: 締切 15 分前のオッズ（確定 × 雑音）と、市場とのブレンド確率で期待値を確認し、tau 未満の買い目を落とす。"""
    act = TICK_ACT[year][["race_id", "type", "combo", "p_comb", "odds"]].rename(columns={"p_comb": "p_blend", "odds": "odds_act"})
    m = dte[["race_id", "type", "combo"]].merge(act, on=["race_id", "type", "combo"], how="left")
    rng = np.random.default_rng(seed)
    o15 = m["odds_act"].to_numpy() * np.exp(rng.normal(0, sigma, len(m))) if sigma > 0 else m["odds_act"].to_numpy()
    ok = (m["p_blend"].to_numpy() * o15 >= tau)
    keep = pol.pack_column(pk, dte.assign(_k=np.where(np.isnan(ok.astype(float)) | ~np.isfinite(m["p_blend"].to_numpy() * o15), 0.0, ok.astype(float))), "_k")
    return w * np.nan_to_num(keep, nan=0.0)


def run_period(tr_years, te_year, seeds, n_boot=500):
    pk, dte, w = rl_alloc(tr_years, te_year, seeds)
    allocs = {"RL（予測オッズ）": w, **baselines(pk, dte)}
    for sg in (0.0, 0.1):
        for tau in (0.85, 1.0):
            allocs[f"RL → T-15確認（σ={sg}, EV≥{tau}）"] = confirm(pk, dte, w, te_year, sg, tau)
    rows = []
    for k, a in allocs.items():
        s = pol.summarize(pol.settle(pk, a), n_boot=n_boot)
        rows.append({"方策": k, **{c: s[c] for c in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation")}})
    # 選ばれた買い目の 予測オッズ ÷ 確定オッズ（賭け金で重みづけ）: 1 を超えれば、予測が実際より高く、判断が楽観的
    act = TICK_ACT[te_year][["race_id", "type", "combo", "odds"]].rename(columns={"odds": "odds_act"})
    m = dte[["race_id", "type", "combo", "odds"]].merge(act, on=["race_id", "type", "combo"], how="left")
    ratio = pol.pack_column(pk, dte.assign(_r=(m["odds"] / m["odds_act"]).to_numpy()), "_r")
    ww = np.where(pk.mask & np.isfinite(ratio), w, 0.0)
    bias = float((ww * np.nan_to_num(ratio, nan=0.0).clip(0, 10)).sum() / max(ww.sum(), 1e-12)) if np.isfinite(ratio).any() else np.nan
    return pd.DataFrame(rows), bias, pk, allocs


FOLDS = [("A", ["2022"], "2023"), ("B", ["2022", "2023"], "2024")] if not SMOKE else [("B", ["2023"], "2024")]
VAL = []
for fold, tr_y, va_y in FOLDS:
    t0 = time.time()
    r, bias, _, _ = run_period(tr_y, va_y, SEEDS)
    r.insert(0, "fold", fold); r["選ばれた買い目の予測÷確定オッズ"] = bias
    VAL.append(r); print(fold, f"{time.time() - t0:.0f}s 予測÷確定 {bias:.3f}", flush=True)
VAL = pd.concat(VAL, ignore_index=True)
display(VAL.pivot_table(index="方策", columns="fold", values="roi").round(3)); VAL.to_csv(OUT / "valid_compare.csv", index=False)

## tp-12（固定パラメータの 1 本）との比較（検証年のみ）


In [ ]:
v1 = BET / "pred_odds" / "pred_v1" / "valid_compare.csv"
cmpt = VAL.pivot_table(index="方策", columns="fold", values="roi").add_prefix("V8_")
if v1.exists():
    old = pd.read_csv(v1).pivot_table(index="方策", columns="fold", values="roi").add_prefix("tp12_")
    cmpt = cmpt.join(old, how="outer")
    for f in ("A", "B"):
        if f"V8_{f}" in cmpt and f"tp12_{f}" in cmpt:
            cmpt[f"差_{f}"] = cmpt[f"V8_{f}"] - cmpt[f"tp12_{f}"]
display(cmpt.round(3)); cmpt.to_csv(OUT / "compare_with_tp12.csv")
bias = VAL.groupby("fold")["選ばれた買い目の予測÷確定オッズ"].first().round(3).to_dict()
print("選ばれた買い目の 予測÷確定オッズ（V8）:", bias)
res = {"evaluated_at": datetime.now().isoformat(timespec="seconds"), "tag": TAG, "git_commit": GIT_COMMIT, "valid_only": True, "odds_prediction_accuracy": acc.to_dict("records"), "valid": VAL.to_dict("records"), "bias": bias,
       "notes": ["test 2025 は使っていない", "単勝の予測オッズは tp-20 の V8（探索パラメータの LightGBM + CatBoost の平均、追い切り評価・厩舎コメント・血統×条件を追加）", "複勝は tp-12 と同じ"]}
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("完了")
